In [0]:
from pyspark.sql import functions as F

CATALOG_SCHEMA = "workspace.ecommerce_de"

# 1. Payment Pipeline Reconciliation Across Layers
bronze_pay_cnt = spark.table(f"{CATALOG_SCHEMA}.bronze_payments").count()
valid_pay_cnt = spark.table(f"{CATALOG_SCHEMA}.staging_valid_payments").count()
quar_pay_cnt = spark.table(f"{CATALOG_SCHEMA}.quarantine_payments").count()
silver_pay_cnt = spark.table(f"{CATALOG_SCHEMA}.silver_payments").count()
fact_pay_cnt = spark.table(f"{CATALOG_SCHEMA}.fact_payments").count()

print("=== PAYMENT LAYER COUNTS ===")
print(f"bronze_payments        : {bronze_pay_cnt}")
print(f"staging_valid_payments : {valid_pay_cnt}")
print(f"quarantine_payments    : {quar_pay_cnt}")
print(f"silver_payments        : {silver_pay_cnt}")
print(f"fact_payments          : {fact_pay_cnt}")

print("\n=== QUARANTINE PAYMENTS BREAKDOWN ===")
spark.table(f"{CATALOG_SCHEMA}.quarantine_payments").groupBy("rejection_reason").count().orderBy(
    F.desc("count")
).show(truncate=False)

print("\n=== SAMPLE BRONZE PAYMENTS (FIRST 5 ROWS) ===")
spark.table(f"{CATALOG_SCHEMA}.bronze_payments").show(5, truncate=False)

# 2. Data Quality & Contract Assertions (dbt-style tests)
checks = []

# Check A: silver_orders primary key uniqueness & non-null
dup_orders = (
    spark.table(f"{CATALOG_SCHEMA}.silver_orders")
    .groupBy("order_id")
    .count()
    .filter((F.col("order_id").isNull()) | (F.col("count") > 1))
    .count()
)
checks.append(("silver_orders_unique_not_null_order_id", dup_orders == 0, f"bad_keys={dup_orders}"))

# Check B: dim_customer has active current rows
active_cust_cnt = (
    spark.table(f"{CATALOG_SCHEMA}.dim_customer")
    .filter(F.col("is_current") == True)
    .count()
)
checks.append(("dim_customer_has_active_rows", active_cust_cnt > 0, f"active_rows={active_cust_cnt}"))

# Check C: fact_orders has no negative or null order_amount
bad_order_amt = (
    spark.table(f"{CATALOG_SCHEMA}.fact_orders")
    .filter((F.col("order_amount").isNull()) | (F.col("order_amount") <= 0))
    .count()
)
checks.append(("fact_orders_positive_amount", bad_order_amt == 0, f"bad_rows={bad_order_amt}"))

# Check D: Orphan order_ids in fact_payments (payments pointing to non-existent orders)
orphan_payments = (
    spark.table(f"{CATALOG_SCHEMA}.fact_payments")
    .alias("p")
    .join(
        spark.table(f"{CATALOG_SCHEMA}.fact_orders").alias("o"),
        on="order_id",
        how="left_anti",
    )
    .count()
)
checks.append(("fact_payments_fk_to_fact_orders", orphan_payments == 0, f"orphan_rows={orphan_payments}"))

# Check E: mart_daily_revenue & mart_customer_360 non-empty
mart_rev_cnt = spark.table(f"{CATALOG_SCHEMA}.mart_daily_revenue").count()
mart_c360_cnt = spark.table(f"{CATALOG_SCHEMA}.mart_customer_360").count()
checks.append(("mart_daily_revenue_populated", mart_rev_cnt > 0, f"rows={mart_rev_cnt}"))
checks.append(("mart_customer_360_populated", mart_c360_cnt > 0, f"rows={mart_c360_cnt}"))

print("\n=== DATA QUALITY ASSERTION RESULTS ===")
failed = []
for name, passed, details in checks:
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name} -> {details}")
    if not passed:
        failed.append(name)

if failed:
    print(f"\n⚠️ Attention needed on checks: {failed}")
else:
    print("\n✅ All core data quality assertions passed!")

In [0]:
spark.table("workspace.ecommerce_de.bronze_payments") \
    .select("event_id", "order_id", "payment") \
    .show(10, truncate=False)